# Maya-CSM TTS Server (Kaggle)

Runs the Maya TTS API on a free Kaggle GPU and exposes it via a Cloudflare tunnel for SillyTavern.

**Before running (all required):**
1. Settings panel (right sidebar) → Accelerator → **GPU T4 x2** (or P100).
2. Settings panel → **Internet → On** (requires a phone-verified Kaggle account).
3. Accept the model terms at https://huggingface.co/sesame/csm-1b with your HF account.
4. Add-ons → Secrets → add `HF_TOKEN` with a Hugging Face read token and attach it to this notebook.
5. Set `REPO_URL` below to your fork/copy of the maya-csm repo.

In [ ]:
REPO_URL = "https://github.com/l0ophole/maya-csm"  # <-- change me

!git clone -q $REPO_URL /kaggle/working/maya-csm
%pip install -q "/kaggle/working/maya-csm[model]"
# Kaggle preinstalls may predate CSM support in transformers
%pip install -q -U transformers peft accelerate torchao

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["MAYA_ADAPTER"] = "shb777/csm-maya-exp2"  # pull LoRA from the Hub
os.environ["MAYA_PRELOAD"] = "1"

In [ ]:
# Start the server in a background thread (model loads now; takes a few minutes first run)
import threading
import uvicorn
from maya_csm.config import Settings
from maya_csm.server import create_app

app = create_app(Settings.from_env())
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()
print("server starting on :8000")

In [ ]:
# Cloudflare tunnel (no account needed). Re-run this cell if the URL dies.
import re
import subprocess
import time

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

proc = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
deadline = time.time() + 30
while time.time() < deadline and url is None:
    line = proc.stdout.readline()
    m = re.search(r"https://[\w.-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
print("Public URL:", url)
print("\nSillyTavern OpenAI-Compatible TTS endpoint:")
print(f"  {url}/v1/audio/speech")

In [ ]:
# In-notebook sanity check (no tunnel needed)
import requests
from IPython.display import Audio

r = requests.post(
    "http://localhost:8000/v1/audio/speech",
    json={"input": "[giggling] Hey there, it's so good to hear your voice.", "voice": "maya"},
)
r.raise_for_status()
Audio(r.content)